<a href="https://colab.research.google.com/github/navirath/ML-lab/blob/main/adaboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
file_path = input("Please enter the path to the file: ")
try:
    with open(file_path, 'r') as file:
        content = file.read()
    # The content is read into the 'content' variable but not displayed.
    # You can add further processing here if needed.
    print(f"File '{file_path}' read successfully. Content is stored in the 'content' variable.")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

KeyboardInterrupt: Interrupted by user

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')
  # The content of the uploaded file is available in uploaded[fn]
  # You can process it here, for example:
  # file_content = uploaded[fn].decode('utf-8')
  # print(file_content)
  # If you want to save it to a local file within the Colab environment:
  # with open(fn, 'wb') as f:
  #   f.write(uploaded[fn])

print("Files uploaded successfully.")

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Load the iris.csv file into a pandas DataFrame
iris_df = pd.read_csv('iris.csv')

# Display the first 5 rows of the DataFrame
display(iris_df.head())

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.estimators = []
        self.estimator_weights = []

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize weights for each sample uniformly
        sample_weights = np.full(n_samples, (1 / n_samples))

        for _ in range(self.n_estimators):
            # Train a weak learner (Decision Stump is common)
            # Set max_depth=1 for a decision stump
            weak_learner = DecisionTreeClassifier(max_depth=1, random_state=42)
            weak_learner.fit(X, y, sample_weight=sample_weights)
            predictions = weak_learner.predict(X)

            # Calculate error rate
            incorrect = (predictions != y)
            error = np.sum(sample_weights[incorrect])

            # Calculate estimator weight (alpha)
            # Avoid division by zero or log of zero/negative
            if error == 0:
                alpha = np.inf # strong learner, essentially perfect
            elif error >= 1.0:
                alpha = -np.inf # worse than random, likely sign error in error calculation or bad model
            else:
                alpha = 0.5 * np.log((1.0 - error) / (error + 1e-10)) # Add small epsilon to avoid log(0)


            # Store estimator and its weight
            self.estimators.append(weak_learner)
            self.estimator_weights.append(alpha)

            # Update sample weights
            # Only update if the estimator is not perfect and not worse than random
            if np.isinf(alpha) or error >= 1.0:
                break # stop if perfect or terrible learner

            # Correctly classified samples have their weights decreased
            # Misclassified samples have their weights increased
            sample_weights *= np.exp(-alpha * y * predictions) # y and predictions must be -1 or 1

            # Normalize weights to sum to 1
            sample_weights /= np.sum(sample_weights)

    def predict(self, X):
        # Combine predictions from all weak learners
        # Weighted majority vote
        final_predictions = np.zeros(X.shape[0])
        for i, estimator in enumerate(self.estimators):
            final_predictions += self.estimator_weights[i] * estimator.predict(X)
        return np.sign(final_predictions)

# Example Usage:
# Generate a synthetic dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=10, random_state=42)

# Convert labels to -1 and 1 for AdaBoost calculation (common practice)
y[y == 0] = -1

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the AdaBoost model
adaboost_model = AdaBoost(n_estimators=10)
adaboost_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = adaboost_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"AdaBoost Model Accuracy: {accuracy:.4f}")

# You can also use sklearn's AdaBoostClassifier for comparison or more advanced features
from sklearn.ensemble import AdaBoostClassifier

# Re-generate data for sklearn's model as it typically expects 0 and 1 for classification
X_sk, y_sk = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=10, random_state=42)
X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)

sk_adaboost = AdaBoostClassifier(n_estimators=10, random_state=42)
sk_adaboost.fit(X_train_sk, y_train_sk)
y_pred_sk = sk_adaboost.predict(X_test_sk)

sk_accuracy = accuracy_score(y_test_sk, y_pred_sk)
print(f"Scikit-learn AdaBoost Accuracy: {sk_accuracy:.4f}")

AdaBoost Model Accuracy: 0.8550
Scikit-learn AdaBoost Accuracy: 0.8550
